# 10.4 Modifying models

In [1]:
# install dependencies
%pip install -q amplpy pandas

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Several commands are provided to help you make limited changes to the current model, without modifying the model file or issuing a full reset. This section describes commands that completely remove or redefine model components, and that temporarily drop constraints, fix variables, or relax integrality restrictions on variables.

## 10.4.1 Removing or redefining model components

The AMPL `delete` command removes a previously declared model component, provided that no other components use it in their declarations. The form of the command is simply `delete` followed by a comma-separated list of names of model components. amplpy has no method of its own for this command, so it is given through `eval()`. Here we read the diet model `dietu.mod` of [Section 10.3](10_3_modifying_data.ipynb) together with the second objective `Total_Number` of [Section 10.2](10_2_setting_up_and_solving_models_and_data.ipynb), and then delete that objective and the `Diet_Min` constraints. The objectives and constraints that remain are listed by `get_objectives()` and `get_constraints()`:

In [2]:
%%writefile dietu.mod

set MINREQ;                     # nutrients with minimum requirements
set MAXREQ;                     # nutrients with maximum requirements
set NUTR = MINREQ union MAXREQ; # nutrients
set FOOD;                       # foods

param cost {FOOD} > 0;
param f_min {FOOD} >= 0;
param f_max {j in FOOD} >= f_min[j];

param n_min {MINREQ} >= 0;
param n_max {MAXREQ} >= 0;

param amt {NUTR,FOOD} >= 0;

var Buy {j in FOOD} >= f_min[j], <= f_max[j];

minimize Total_Cost: sum {j in FOOD} cost[j] * Buy[j];

subject to Diet_Min {i in MINREQ}:
   sum {j in FOOD} amt[i,j] * Buy[j] >= n_min[i];

subject to Diet_Max {i in MAXREQ}:
   sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];

Writing dietu.mod


In [3]:
%%writefile number_obj.mod

minimize Total_Number: sum {j in FOOD} Buy[j];

Writing number_obj.mod


In [4]:
ampl.read("dietu.mod")
ampl.read("number_obj.mod")

ampl.eval("delete Total_Number, Diet_Min;")
print("objectives: ", [name for name, _ in ampl.get_objectives()])
print("constraints:", [name for name, _ in ampl.get_constraints()])

objectives:  ['Total_Cost']
constraints: ['Diet_Max']


Normally you cannot `delete` a set, parameter, or variable, because it is declared for use later in the model; but you can `delete` any objective or constraint. You can also specify a component "name" of the form `check n` to `delete` the nth `check` statement in the current model.

The `purge` command has the same form, but with the keyword `purge` in place of `delete`. It removes not only the listed components, but also all components that depend on them either directly (by referring to them) or indirectly (by referring to their dependents). Thus for example in `diet.mod` we have

```ampl
param f_min {FOOD} >= 0;
param f_max {j in FOOD} >= f_min[j];
var Buy {j in FOOD} >= f_min[j], <= f_max[j];
minimize Total_Cost: sum {j in FOOD} cost[j] * Buy[j];
```

The command `purge f_min` deletes parameter `f_min` and the components whose declarations refer to `f_min`, including parameter `f_max` and variable `Buy`. It also deletes objective `Total_Cost`, which depends indirectly on `f_min` through its reference to `Buy`.

If you're not sure which components depend on some given component, you can use the `xref` command to find out; in amplpy, every model component has an `xref()` method that returns the names of its dependents:

In [5]:
%%writefile diet.mod

set NUTR;
set FOOD;

param cost {FOOD} > 0;
param f_min {FOOD} >= 0;
param f_max {j in FOOD} >= f_min[j];

param n_min {NUTR} >= 0;
param n_max {i in NUTR} >= n_min[i];

param amt {NUTR,FOOD} >= 0;

var Buy {j in FOOD} >= f_min[j], <= f_max[j];

minimize Total_Cost:  sum {j in FOOD} cost[j] * Buy[j];

subject to Diet {i in NUTR}:
   n_min[i] <= sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];

Writing diet.mod


In [6]:
ampl.reset()
ampl.read("diet.mod")

ampl.param["f_min"].xref()

['f_max', 'Buy', 'Total_Cost', 'Diet']

Like `delete` and `purge`, the `xref` command can be applied to any list of model components. Purging `f_min` leaves only the components that do not depend on it:

In [7]:
ampl.eval("purge f_min;")

print("sets:       ", [name for name, _ in ampl.get_sets()])
print("parameters: ", [name for name, _ in ampl.get_parameters()])
print("variables:  ", [name for name, _ in ampl.get_variables()])
print("objectives: ", [name for name, _ in ampl.get_objectives()])
print("constraints:", [name for name, _ in ampl.get_constraints()])

sets:        ['NUTR', 'FOOD']
parameters:  ['cost', 'n_min', 'n_max', 'amt']
variables:   []
objectives:  []
constraints: []


Once a component has been removed by `delete` or `purge`, any previously hidden meaning of the component's name becomes visible again. After a constraint named `prod` is deleted, for instance, AMPL again recognizes `prod` as an iterated multiplication operator ([Table 7-1](../07/7_2_arithmetic_expressions.ipynb#table-7-1)).

If there is no previously hidden meaning, the name of a component removed by `delete` or `purge` becomes again unused, and may subsequently be declared as the name of any new component of any type. If you only want to make some relatively limited modifications to a declaration, however, then you will probably find `redeclare` to be more convenient. You can change any component's declaration by writing the keyword `redeclare` followed by the complete revised declaration that you would like to substitute. Looking again at `diet.mod`, for example,

In [8]:
ampl.reset()
ampl.read("diet.mod")

ampl.eval("redeclare param f_min {FOOD} > 0 integer;")
print(ampl.param["f_min"])   # the current declaration of f_min

param f_min{FOOD}  integer > 0;


changes only the validity conditions on `f_min`. The declarations of all components that depend on `f_min` are left unchanged, as are any values previously assigned to `f_min`.

A list of all component types to which `delete`, `purge`, `xref`, and `redeclare` may be applied is given in [Section A.18.5](../appendix_a/a_18_5_changing_a_model_delete_purge_redeclare.ipynb). After any of these commands, amplpy refreshes its lists of model components (`ampl.var`, `ampl.param`, and so on) the next time they are used, so that they always reflect the current model.

## 10.4.2 Changing the model: fix, unfix; drop, restore

The simplest (but most drastic) way to change the model is by calling `reset()`, which expunges all of the current model and data. Following `reset()`, you can call `read()` and assign data to set up a different optimization problem; the effect is like closing the AMPL session and starting a new one, except that options are not reset to their default values. Since you can edit model files while keeping your Python session active, `reset()` is valuable for debugging and experimentation: you may make changes to the model files, call `reset()`, then read in the modified files.

The AMPL `drop` command instructs AMPL to ignore certain constraints or objectives of the current model. As an example, the constraints of `dietu.mod` include

```ampl
subject to Diet_Max {i in MAXREQ}:
   sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];
```

We set up this model again, with its data:

In [9]:
import pandas as pd

food_df = pd.DataFrame(
    [
        ("BEEF", 3.19, 2, 10),
        ("CHK",  2.59, 2, 10),
        ("FISH", 2.29, 2, 10),
        ("HAM",  2.89, 2, 10),
        ("MCH",  1.89, 2, 10),
        ("MTL",  1.99, 2, 10),
        ("SPG",  1.99, 2, 10),
        ("TUR",  2.49, 2, 10),
    ],
    columns=["FOOD", "cost", "f_min", "f_max"],
).set_index("FOOD")

MINREQ = ["A", "B1", "B2", "C", "CAL"]
MAXREQ = ["A", "NA", "CAL"]

n_min = {"A": 700, "B1": 0, "B2": 0, "C": 700, "CAL": 16000}
n_max = {"A": 20000, "NA": 50000, "CAL": 24000}

# Nutrient content: one row per nutrient, one column per food
amt_df = pd.DataFrame(
    {
        "BEEF": [60, 20, 10, 15,  938, 295],
        "CHK":  [ 8,  0, 20, 20, 2180, 770],
        "FISH": [ 8, 10, 15, 10,  945, 440],
        "HAM":  [40, 40, 35, 10,  278, 430],
        "MCH":  [15, 35, 15, 15, 1182, 315],
        "MTL":  [70, 30, 15, 15,  896, 400],
        "SPG":  [25, 50, 25, 15, 1329, 370],
        "TUR":  [60, 20, 15, 10, 1397, 450],
    },
    index=["A", "C", "B1", "B2", "NA", "CAL"],
)
amt = amt_df.stack().to_dict()   # {(nutrient, food): amount}

In [10]:
ampl.reset()
ampl.read("dietu.mod")

ampl.set_data(food_df, "FOOD")
ampl.set["MINREQ"] = MINREQ
ampl.set["MAXREQ"] = MAXREQ
ampl.param["n_min"] = n_min
ampl.param["n_max"] = n_max
ampl.param["amt"] = amt

In amplpy, every constraint and objective has `drop()` and `restore()` methods. They apply to all members of an indexed collection, so that the entire collection of `Diet_Max` constraints can be ignored by

In [11]:
ampl.con["Diet_Max"].drop()
ampl.display("{i in MAXREQ} Diet_Max[i].astatus")

Diet_Max[i].astatus [*] :=
  A  drop
CAL  drop
 NA  drop
;



The `restore()` method reverses the effect of `drop()`:

In [12]:
ampl.con["Diet_Max"].restore()

To ignore a particular one of these constraints, give the AMPL `drop` command through `eval()`:

In [13]:
ampl.eval("drop Diet_Max['CAL'];")
ampl.display("{i in MAXREQ} Diet_Max[i].astatus")

Diet_Max[i].astatus [*] :=
  A  pre
CAL  drop
 NA  in
;



In general, this command consists of the keyword `drop`, an optional indexing expression, and a constraint name that may be subscripted, as in `drop {i in MAXNOT} Diet_Max[i];` for some subset `MAXNOT` of `MAXREQ`. Successive `drop` commands and calls have a cumulative effect. The `restore` command reverses the effect of `drop`. It has the same syntax, except for the keyword `restore`:

In [14]:
ampl.eval("restore Diet_Max['CAL'];")

The `fix()` method of a variable fixes it at its current value, as if there were a constraint that the variable must equal this value; the `unfix()` method reverses the effect. Like `drop()` and `restore()`, these methods apply to all members of an indexed variable, but they may also be applied to individual members, selected with a subscript. For example, here we initialize all variables of our diet problem to their lower bounds, fix all variables representing foods that have more than 1200 mg of sodium per package, and optimize over the remaining variables:

In [15]:
ampl.var["Buy"] = food_df["f_min"].to_dict()

high_sodium = [j for j in food_df.index if amt["NA", j] > 1200]
for j in high_sodium:
    ampl.var["Buy"][j].fix()

ampl.solve(solver="highs")
assert ampl.solve_result == "solved", ampl.solve_result

ampl.display('{j in FOOD} (Buy[j].lb, Buy[j], amt["NA",j])')

HiGHS 1.15.1: optimal solution; objective 86.92
1 simplex iterations
0 barrier iterations
:    Buy[j].lb    Buy[j]  amt['NA',j]    :=
BEEF      2       2            938
CHK       2       2           2180
FISH      2      10            945
HAM       2       2            278
MCH       2       9.42857     1182
MTL       2      10            896
SPG       2       2           1329
TUR       2       2           1397
;



Rather than setting and fixing the variables in separate statements, you can give a value as the argument of `fix()`, as in `ampl.var["Buy"][j].fix(food_df.loc[j, "f_min"])`. The `unfix()` method reverses the effect of `fix()`:

In [16]:
ampl.var["Buy"].unfix()

## 10.4.3 Relaxing integrality

Changing option `relax_integrality` from its default of 0 to any nonzero value:

In [17]:
ampl.option["relax_integrality"] = 1

tells AMPL to ignore all restrictions of variables to integer values. Variables declared `integer` get whatever bounds you specified for them, while variables declared `binary` are given a lower bound of zero and an upper bound of one. To restore the integrality restrictions, set the `relax_integrality` option back to 0:

In [18]:
ampl.option["relax_integrality"] = 0

A variable's name followed by the suffix `.relax` indicates its current integrality relaxation status: 0 if integrality is enforced, nonzero otherwise. You can make use of this suffix, through the `set_suffix()` method, to relax integrality on selected variables only. For example,

```python
ampl.var["Buy"]["CHK"].set_suffix("relax", 1)
```

relaxes integrality only on the variable `Buy['CHK']`, while

```python
for j in food_df.index:
    if food_df.loc[j, "f_min"] > allow_frac:
        ampl.var["Buy"][j].set_suffix("relax", 1)
```

relaxes integrality on all `Buy` variables for foods that have a minimum purchase of at least some cutoff value `allow_frac`.

Some of the solvers that work with AMPL provide their own directives for relaxing integrality, but these do not necessarily have the same effect as AMPL's `relax_integrality` option or `.relax` suffix. The distinction is due to the effects of AMPL's problem simplification, or presolve, stage ([Section 14.1](../14/14_1_presolve.ipynb)). AMPL drops integrality restrictions before the presolve phase, so that the solver receives a true continuous relaxation of the original integer problem. If the relaxation is performed by the solver, however, then the integrality restrictions are still in effect during AMPL's presolve phase, and AMPL may perform some additional tightening and simplification as a result.

As a simple example, suppose that diet model variable declarations are written to allow the food limits `f_max` to be adjusted by setting an additional parameter, `scale`:

```ampl
var Buy {j in FOOD} integer >= f_min[j], <= scale * f_max[j];
```

In [19]:
%%writefile dieti.mod

set NUTR;
set FOOD;

param cost {FOOD} > 0;
param f_min {FOOD} >= 0;
param f_max {j in FOOD} >= f_min[j];
param scale > 0, default 1;   # scale factor for the upper limits

param n_min {NUTR} >= 0;
param n_max {i in NUTR} >= n_min[i];

param amt {NUTR,FOOD} >= 0;

var Buy {j in FOOD} integer >= f_min[j], <= scale * f_max[j];

minimize Total_Cost:  sum {j in FOOD} cost[j] * Buy[j];

subject to Diet {i in NUTR}:
   n_min[i] <= sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];

Writing dieti.mod


In our example of [Figure 2-3](../02/2_3_using_the_AMPL_diet_model.ipynb#fig-2-3), all of the `f_max` values are 10; suppose that also we set `scale` to 0.95. First, here are the results of solving the unrelaxed problem:

In [20]:
import pandas as pd

food_df = pd.DataFrame(
    [
        ("BEEF", 3.19, 2, 10),
        ("CHK",  2.59, 2, 10),
        ("FISH", 2.29, 2, 10),
        ("HAM",  2.89, 2, 10),
        ("MCH",  1.89, 2, 10),
        ("MTL",  1.99, 2, 10),
        ("SPG",  1.99, 2, 10),
        ("TUR",  2.49, 2, 10),
    ],
    columns=["FOOD", "cost", "f_min", "f_max"],
).set_index("FOOD")

nutr_df = pd.DataFrame(
    [
        ("A",     700, 20000),
        ("C",     700, 20000),
        ("B1",    700, 20000),
        ("B2",    700, 20000),
        ("NA",      0, 50000),
        ("CAL", 16000, 24000),
    ],
    columns=["NUTR", "n_min", "n_max"],
).set_index("NUTR")

# Nutrient content: one row per nutrient, one column per food
amt_df = pd.DataFrame(
    {
        "BEEF": [60, 20, 10, 15,  938, 295],
        "CHK":  [ 8,  0, 20, 20, 2180, 770],
        "FISH": [ 8, 10, 15, 10,  945, 440],
        "HAM":  [40, 40, 35, 10,  278, 430],
        "MCH":  [15, 35, 15, 15, 1182, 315],
        "MTL":  [70, 30, 15, 15,  896, 400],
        "SPG":  [25, 50, 25, 15, 1329, 370],
        "TUR":  [60, 20, 15, 10, 1397, 450],
    },
    index=["A", "C", "B1", "B2", "NA", "CAL"],
)

In [21]:
ampl.reset()
ampl.read("dieti.mod")

ampl.set_data(food_df, "FOOD")
ampl.set_data(nutr_df, "NUTR")
ampl.param["amt"] = amt_df.stack().to_dict()

ampl.param["scale"] = 0.95
print("relax_integrality =", ampl.option["relax_integrality"])

ampl.solve(solver="highs")
assert ampl.solve_result == "solved", ampl.solve_result

relax_integrality = 0
HiGHS 1.15.1: optimal solution; objective 122.89
7 simplex iterations
1 branching nodes


When no relaxation is specified in AMPL, presolve sees that all the variables have upper limits of 9.5, and since it knows that the variables must take integer values, it rounds these limits down to 9. Then these limits are sent to the solver, where they remain even if we specify a solver directive for integrality relaxation, such as the HiGHS option `relax=1`:

In [22]:
ampl.solve(solver="highs", highs_options="relax=1")
assert ampl.solve_result == "solved", ampl.solve_result

ampl.display("Buy")

HiGHS 1.15.1:   alg:relax = 1
HiGHS 1.15.1: optimal solution; objective 120.2421057
2 simplex iterations
0 barrier iterations
Buy [*] :=
BEEF  8.39898
 CHK  2
FISH  2
 HAM  9
 MCH  9
 MTL  9
 SPG  8.93436
 TUR  2
;



If instead option `relax_integrality` is set to 1, presolve leaves the upper limits at 9.5 and sends those to the solver, with the result being a less constrained problem and hence a lower objective value. (Solver options given to `solve()` remain in effect for later solves, so we first clear the `relax` directive.)

In [23]:
ampl.option["highs_options"] = ""
ampl.option["relax_integrality"] = 1

ampl.solve(solver="highs")
assert ampl.solve_result == "solved", ampl.solve_result

ampl.display("Buy")

HiGHS 1.15.1: optimal solution; objective 119.1507545
0 simplex iterations
0 barrier iterations
Buy [*] :=
BEEF  6.8798
 CHK  2
FISH  2
 HAM  9.5
 MCH  9.5
 MTL  9.5
 SPG  9.1202
 TUR  2
;



Variables that were at upper bound 9 in the previous solution are now at upper bound 9.5.

The same situation can arise in much less obvious circumstances, and can lead to unexpected results. In general, the optimal value of an integer program under AMPL's `relax_integrality` option may be lower (for minimization) or higher (for maximization) than the optimal value reported by the solver's relaxation directive.